In [2]:
# Cell 1 - Imports
import numpy as np
import pandas as pd
import pickle
import os
from scipy.stats import multivariate_normal

In [3]:
# Cell 2 - Load the dataset
df = pd.read_csv('../data/dataset.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nLabel counts:")
print(df['label'].value_counts())

Shape: (1100, 5)

First 5 rows:
       width    spacing    x_coord    y_coord  label
0  41.968918  55.415902  -5.577101  10.044452    0.0
1  49.230640  45.360199   8.829402 -10.758954    0.0
2  57.787601  41.036929  -1.923631   8.367453    0.0
3  48.634041  50.051837 -13.665386   0.405630    0.0
4  48.305596  47.094536  10.231907  -2.229131    0.0

Label counts:
label
0.0    1000
1.0     100
Name: count, dtype: int64


In [6]:
# Cell 3 - Load Gaussian parameters from Member 1
with open('../results/gaussian_params.pkl', 'rb') as f:
    gaussian_params = pickle.load(f)

print("Keys in gaussian_params:", list(gaussian_params.keys()))
print("\nFull contents:")
for key, value in gaussian_params.items():
    print(f"\n{key}:")
    print(value)

Keys in gaussian_params: ['safe', 'hotspot']

Full contents:

safe:
{'mu': array([4.98264012e+01, 4.98625513e+01, 4.21842398e-01, 3.25703004e-02]), 'sigma': array([[ 23.50757942,  -1.2013575 ,   0.12420324,  -0.53014293],
       [ -1.2013575 ,  26.93732452,   0.24680065,   0.7663428 ],
       [  0.12420324,   0.24680065, 104.83786786,   8.29489386],
       [ -0.53014293,   0.7663428 ,   8.29489386, 105.28018563]])}

hotspot:
{'mu': array([30.34232607, 20.06189587,  3.4498625 ,  1.51184713]), 'sigma': array([[ 19.73329459,   1.08084747,  -7.48093751, -13.09261654],
       [  1.08084747,  24.75242146,   0.25442665,   4.71597423],
       [ -7.48093751,   0.25442665, 112.11597123, -16.91687942],
       [-13.09261654,   4.71597423, -16.91687942,  95.79778764]])}


In [7]:
# Cell 4 - Extract parameters from Member 1's structure
mu_safe = gaussian_params['safe']['mu']
sigma_safe = gaussian_params['safe']['sigma']

mu_hotspot = gaussian_params['hotspot']['mu']
sigma_hotspot = gaussian_params['hotspot']['sigma']

print("mu_safe:", mu_safe)
print("mu_hotspot:", mu_hotspot)
print("\nsigma_safe shape:", sigma_safe.shape)
print("sigma_hotspot shape:", sigma_hotspot.shape)
print("\nAll parameters loaded successfully! ✅")

mu_safe: [4.98264012e+01 4.98625513e+01 4.21842398e-01 3.25703004e-02]
mu_hotspot: [30.34232607 20.06189587  3.4498625   1.51184713]

sigma_safe shape: (4, 4)
sigma_hotspot shape: (4, 4)

All parameters loaded successfully! ✅


In [8]:
# Cell 5 - Separate features and labels

X = df[['width', 'spacing', 'x_coord', 'y_coord']].values
y = df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nSample X[0]:", X[0])
print("Sample y[0]:", y[0])

X shape: (1100, 4)
y shape: (1100,)

Sample X[0]: [41.96891757 55.41590154 -5.57710131 10.04445157]
Sample y[0]: 0.0


In [9]:
# Cell 6 - Compute priors

num_samples = len(y)

num_hotspot = np.sum(y == 1)
num_safe = np.sum(y == 0)

P_hotspot = num_hotspot / num_samples
P_safe = num_safe / num_samples

print("P(hotspot):", P_hotspot)
print("P(safe):", P_safe)

P(hotspot): 0.09090909090909091
P(safe): 0.9090909090909091


In [10]:
# Cell 7 - Create Gaussian distributions

gaussian_safe = multivariate_normal(
    mean=mu_safe,
    cov=sigma_safe,
    allow_singular=True
)

gaussian_hotspot = multivariate_normal(
    mean=mu_hotspot,
    cov=sigma_hotspot,
    allow_singular=True
)

print("Gaussian models created successfully ✅")

Gaussian models created successfully ✅


In [11]:
# Cell 8 - Test likelihood on a single sample

x_sample = X[0]

likelihood_safe = gaussian_safe.logpdf(x_sample)
likelihood_hotspot = gaussian_hotspot.logpdf(x_sample)

print("x_sample:", x_sample)

print("\nlog P(x | safe):", likelihood_safe)
print("log P(x | hotspot):", likelihood_hotspot)

x_sample: [41.96891757 55.41590154 -5.57710131 10.04445157]

log P(x | safe): -14.024500434708797
log P(x | hotspot): -39.63270658795179


In [12]:
# Cell 9 - Convert priors to log space

log_P_hotspot = np.log(P_hotspot)
log_P_safe = np.log(P_safe)

print("log P(hotspot):", log_P_hotspot)
print("log P(safe):", log_P_safe)

log P(hotspot): -2.3978952727983707
log P(safe): -0.0953101798043249


In [13]:
# Cell 10 - Compute log posterior for one sample

# Already computed:
# likelihood_safe
# likelihood_hotspot

log_posterior_safe = likelihood_safe + log_P_safe
log_posterior_hotspot = likelihood_hotspot + log_P_hotspot

print("log posterior (safe):", log_posterior_safe)
print("log posterior (hotspot):", log_posterior_hotspot)

log posterior (safe): -14.119810614513122
log posterior (hotspot): -42.03060186075016


In [14]:
# Cell 11 - Convert log posterior to probability

# Stack values
log_vals = np.array([log_posterior_safe, log_posterior_hotspot])

# Stability trick
max_log = np.max(log_vals)
log_vals_shifted = log_vals - max_log

# Convert back to probabilities
probs = np.exp(log_vals_shifted)
probs = probs / np.sum(probs)

P_safe_given_x = probs[0]
P_hotspot_given_x = probs[1]

print("P(safe | x):", P_safe_given_x)
print("P(hotspot | x):", P_hotspot_given_x)

P(safe | x): 0.9999999999992439
P(hotspot | x): 7.559574932000147e-13
